# Window features for the time-window experiment

This notebook builds the 90 features of `Part3_Feature_Engineering.ipynb` from each user's history cut at **7 days** and at **90 days** before the search query. The two resulting feature sets are used by the time-window experiment in `Elysia_Results_and_Experiments.ipynb` (Section 7.1), which compares a model trained on 7 days of history, on 90 days, and on the full year. They are written to `processed/lgbm_features_window_7d/` and `processed/lgbm_features_window_90d/`, in the same layout as the full-year features in `processed/lgbm_features_all/`. The full-year files are not changed.

Steps 0 to 5 are the feature pipeline of `Part3_Feature_Engineering.ipynb`, with the same settings and the same `build_chunk_features` function, so that the window features are computed by exactly the same code as the full-year features and differ only in the history they see. Step 6 does the cutting and the writing. The notebook needs only the saved intermediates of `Part3_Feature_Engineering.ipynb` in `processed/part3_features/intermediates_backup/` (session chunks, split, product tables, item2vec and term2vec vectors, training-set priors), which Steps 1 to 4 check and load. The item2vec and term2vec vectors are reused as they are, because they describe products, not a user's history, so a shorter history does not change them.

In [10]:
%pip install -q gensim==4.4.0

## Step 0 — Setup

All settings below are the same as in `Part3_Feature_Engineering.ipynb`, because the saved intermediates were built with them and the features must be computed the same way. `SAVE_REBUILT_TO_DRIVE` controls whether anything rebuilt here (if a saved file is missing or inconsistent) is copied back to the backup folder. `OUT_DIR`, `CHUNK_DIR`, `MERGE_DONE` and `DELETE_EACH_CHUNK_AFTER_MERGE` belong to the full-year pipeline and are not used by Step 6, which writes to its own window folders.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import gc, json, math, os, shutil, time
from collections import Counter
from itertools import chain
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import scipy.sparse as sp
from sklearn.preprocessing import normalize

try:
    import psutil
    def ram(label=""):
        print(f"[RAM] {label}: {psutil.Process().memory_info().rss / 1e9:.2f} GB in use")
except ImportError:
    def ram(label=""):
        pass

# ---------------- paths ----------------
PROJECT_DIR = Path("/content/drive/MyDrive/BT4222 Final/JDSearch")
RAW_DIR = PROJECT_DIR / "Raw Data"
USER_TXT = RAW_DIR / "user_behavior_data.txt"
PRODUCT_TXT = RAW_DIR / "product_meta_data.txt"
PRODUCT_CACHE_DIR = RAW_DIR / "product_chunks"
OUTPUT_DIR = PROJECT_DIR / "processed"
BACKUP_DIR = OUTPUT_DIR / "part3_features" / "intermediates_backup"   # saved intermediates of Part3_Feature_Engineering.ipynb
OUT_DIR = OUTPUT_DIR / "lgbm_features_all"                           # the full-year features (not written by this notebook)
CHUNK_DIR = OUT_DIR / "chunks"                                       # per-chunk features (not used in this notebook)
MERGE_DONE = OUT_DIR / "MERGE_COMPLETE.json"
P3_DIR = OUT_DIR                                                     # name used by the feature dictionary code
LOCAL_DIR = Path("/content/part5_local")                             # local working copies
STAGE_DIR, ARR_DIR, TITLE_DIR = LOCAL_DIR / "stage", LOCAL_DIR / "arrays", LOCAL_DIR / "titles"
for d in [OUT_DIR, CHUNK_DIR, STAGE_DIR, ARR_DIR, TITLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SAVE_REBUILT_TO_DRIVE = True          # copy anything rebuilt here back to BACKUP_DIR
DELETE_EACH_CHUNK_AFTER_MERGE = True  # not used in this notebook

# ---------------- settings: identical to Part3_Feature_Engineering ----------------
SEED = 42
SESSIONS_PER_CHUNK = 2000
READ_ROWS = 10_000
CART_MAX_DAYS = 90
TYPE_CODE = {"CLICK": 0, "CART": 1, "ORD": 2, "FLW": 3}
TYPE_NAMES = ["click", "cart", "ord", "flw"]
TYPE_WEIGHT = np.array([1.0, 3.0, 5.0, 2.0])
TAU_DAYS = 30.0
TERM_SEP = "\x18"
TITLE_MIN_DF = 5
EMB_DIM, EMB_WINDOW, EMB_EPOCHS = 64, 5, 5
I2V_MIN_COUNT = 5
PRIOR_ALPHA = 20.0
N_FOLDS = 5
BURST_GAP_DAYS = 30 / 1440
SCREEN_SESSIONS = 10_000
EXPECTED = {"sessions": 173_831, "eligible": 171_175, "candidate_rows": 15_348_049, "carts_removed": 1_177}


# ---------------- backup helpers ----------------
def restore(*names):
    # Copy files/folders from the Drive backup to local disk where missing; True if all exist locally.
    ok = True
    for n in names:
        dst, src = LOCAL_DIR / n, BACKUP_DIR / n
        if src.is_dir():
            for f in src.rglob("*"):
                t = dst / f.relative_to(src)
                if f.is_file() and not t.exists():
                    t.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(f, t)
        elif src.exists() and not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
        ok = ok and dst.exists() and (not dst.is_dir() or any(dst.iterdir()))
    return ok


def backup(*names):
    # Copy rebuilt files/folders from local disk to the Drive backup (if SAVE_REBUILT_TO_DRIVE).
    if not SAVE_REBUILT_TO_DRIVE:
        return
    for n in names:
        src, dst = LOCAL_DIR / n, BACKUP_DIR / n
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
        elif src.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)


def mark_done(flag):
    (LOCAL_DIR / f"{flag}.done").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))


ram("setup")

Mounted at /content/drive
[RAM] setup: 0.24 GB in use


### The feature list

The 90 features, grouped by family, are defined here exactly as in `Part3_Feature_Engineering.ipynb`. The feature computation and the saved column order both refer to this list, so the window files have the same columns in the same order as the full-year files.

In [2]:
ATTRS = ["brand", "shop", "cate1", "cate2", "cate3", "cate4"]       # order of the attribute columns
ATTR_SOURCE = ["brand_id", "shop_id", "cate_id_1", "cate_id_2", "cate_id_3", "cate_id_4"]
BRAND, CATE3 = 0, 4                                                  # column positions used by E, F, G, H
H_BASE = ["brand_aff", "shop_aff", "cate3_aff", "title_cos_profile", "title_cos_max", "i2v_cos_max", "item_ctr_train"]

FAMILIES = {
    "A": ["exact_wid_click_0_7d", "exact_wid_cart_0_7d", "exact_wid_ord_0_7d", "exact_wid_cart_7_90d",
          "exact_wid_ord_7_90d", "exact_wid_ord_90_365d", "exact_wid_last_seen_days"],
    "B": [f"{k}_{s}" for k in ATTRS for s in ["aff", "share", "last_seen_days", "ord_cnt"]],
    "C": ["title_cos_profile", "title_cos_profile_ord", "title_cos_max", "title_hist_qry_overlap"],
    "D": ["i2v_cos_mean", "i2v_cos_max", "i2v_cos_last5", "t2v_cos_profile"],
    "E": ["last_same_brand", "last_same_cate3", "last_title_cos", "last_i2v_cos"],
    "F": ["qhist_n_actions", "qhist_wid_cnt", "qhist_brand_aff", "qhist_cate3_aff"],
    "G": ["item_ctr_train", "brand_ctr_train", "cate3_ctr_train", "item_impr_train"],
    "H": [f"{f}_slate_{s}" for f in H_BASE for s in ["pct", "z"]] + ["cate3_slate_share", "slate_size"],
    "context": [f"{t}_count_{w}" for w in ["0_7d", "7_90d", "90_365d"] for t in TYPE_NAMES]
               + ["history_len_clean", "history_span_days", "days_since_last_interaction",
                  "share_click", "share_cart", "share_ord", "share_flw",
                  "cate3_entropy", "n_distinct_brands", "qryless_share", "burst_len"],
}
FEATURE_COLUMNS = [f for fam in FAMILIES.values() for f in fam]
FAMILY_OF = {f: fam for fam, cols in FAMILIES.items() for f in cols}
ID_COLUMNS = ["session_id", "wid", "label", "split"]
assert len(FEATURE_COLUMNS) == len(set(FEATURE_COLUMNS)) == 90
print({k: len(v) for k, v in FAMILIES.items()}, "total", len(FEATURE_COLUMNS))

{'A': 7, 'B': 24, 'C': 4, 'D': 4, 'E': 4, 'F': 4, 'G': 4, 'H': 16, 'context': 23} total 90


### Inventory of the saved intermediates

This cell lists what is available in `intermediates_backup/`, so it is visible before anything runs which steps will reuse saved files and which would have to rebuild them. It only reports; the following steps make the decisions.

In [3]:
PRODUCT_FILES = ["lookup_wid.npy", "lookup_row.npy", "prod_attr.npy", "prod_meta.json", "title_vocab.parquet",
                 "title_indptr.npy", "title_indices.bin", "title_data.bin"]
I2V_FILES = ["i2v_keys.npy", "i2v_vecs.npy"]
T2V_FILES = ["t2v_term_vecs.npy"]


def size_gb(p):
    if not p.exists():
        return None
    return (sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) if p.is_dir() else p.stat().st_size) / 1e9


rows = []
for group, names in [("session chunks", ["stage", "sessions.parquet", "step1_stats.json"]),
                     ("product tables", [f"arrays/{n}" for n in PRODUCT_FILES]),
                     ("item2vec", [f"arrays/{n}" for n in I2V_FILES]),
                     ("term2vec", [f"arrays/{n}" for n in T2V_FILES]),
                     ("title cache (only needed to retrain term2vec)", ["titles"])]:
    for n in names:
        s = size_gb(BACKUP_DIR / n)
        rows.append({"artifact": group, "file": n, "in backup": s is not None, "size (GB)": None if s is None else round(s, 3)})
display(pd.DataFrame(rows))
stage_bk = BACKUP_DIR / "stage"
print("Backed-up chunk files:", {k: len(list(stage_bk.glob(f"{k}_*.parquet"))) if stage_bk.exists() else 0
                                 for k in ["cand", "hist", "priors"]})
print("Raw files present: user file", USER_TXT.exists(), "| product file", PRODUCT_TXT.exists() or PRODUCT_CACHE_DIR.exists())
print("Full-year features present:", MERGE_DONE.exists())

,artifact,file,in backup,size (GB)
0,session chunks,stage,True,0.696
1,session chunks,sessions.parquet,True,0.004
2,session chunks,step1_stats.json,True,0.000
3,product tables,arrays/lookup_wid.npy,True,0.092
4,product tables,arrays/lookup_row.npy,True,0.092
5,product tables,arrays/prod_attr.npy,True,0.552
6,product tables,arrays/prod_meta.json,True,0.000
7,product tables,arrays/title_vocab.parquet,True,0.002
8,product tables,arrays/title_indptr.npy,True,0.092
9,product tables,arrays/title_indices.bin,True,1.073


Backed-up chunk files: {'cand': 86, 'hist': 86, 'priors': 86}
Raw files present: user file True | product file True
Full-year features present: True


## Step 1 — Session chunks and the split

This step loads the session chunks saved by `Part3_Feature_Engineering.ipynb`: candidates (`cand_*`), cleaned histories (`hist_*`), the session table and the Step 1 statistics. It checks that a candidate and a history file exist for every chunk, and that the number of candidate rows matches the statistics. Only if a check fails are the chunks rebuilt from `user_behavior_data.txt`, with the same code as that notebook, which:
- parses the candidates and labels and keeps the eligible sessions (those with at least one engaged and one non-engaged candidate);
- shuffles each session's candidates with seed 42, because the raw lists store engaged products first;
- parses and cleans the histories (elapsed days before the query; carts older than 90 days removed);
- writes chunks of 2,000 sessions.

The seed is fixed, so a rebuild produces exactly the same chunks. The 80/10/10 split is then reproduced with the same method, seed and ordering as `Siqian_Data_Preprocessing.ipynb`, so the split used here is identical to the one every other notebook uses.

In [4]:
def split_flat(series, sep):
    # Split each string of a Series on `sep`; return the flat token array and the number of tokens per row.
    vals = series.fillna("").to_numpy(dtype=object)
    lists = [v.split(sep) if v else [] for v in vals]
    lens = np.fromiter((len(l) for l in lists), dtype=np.int64, count=len(lists))
    flat = np.array(list(chain.from_iterable(lists)), dtype=object)
    return flat, lens


def within_pos(lens):
    # Position of each token within its own row (0, 1, 2, ... restarting for every row).
    starts = np.cumsum(lens) - lens
    return np.arange(lens.sum(), dtype=np.int64) - np.repeat(starts, lens)


def to_int64(tokens, what):
    # Convert ID tokens to int64; stop with a clear message if any ID is not an integer.
    s = pd.to_numeric(pd.Series(tokens, dtype=object), errors="coerce")
    if s.isna().any():
        bad = pd.Series(tokens)[s.isna()].head(3).tolist()
        raise ValueError(f"Non-integer {what} found, e.g. {bad}. The notebook stores IDs as integers to save memory.")
    return s.to_numpy(dtype=np.int64)


def select_first(flat, lens, keep_n):
    # Keep the first keep_n[i] tokens of row i (the truncation that Siqian_Data_Preprocessing's zip performs).
    pos = within_pos(lens)
    return flat[pos < np.repeat(keep_n, lens)]


def parse_block(df, sid0):
    # Parse one block of raw rows. Returns candidate rows, cleaned history rows and session rows
    # for the eligible sessions only, plus counters for the printed checks.
    n = len(df)
    local = np.arange(n)
    stats = Counter()

    # ---- candidates and eligibility (as in Siqian_Data_Preprocessing) ----
    cw, cn = split_flat(df["candidate_wid_list"], "_")
    cl, cln = split_flat(df["candidate_label_list"], "_")
    lab = pd.to_numeric(pd.Series(cl, dtype=object)).to_numpy()
    l_local = np.repeat(local, cln)
    n_pos = np.bincount(l_local, weights=(lab > 0), minlength=n)
    n_neg = np.bincount(l_local, weights=(lab == 0), minlength=n)
    eligible = (n_pos > 0) & (n_neg > 0)
    # as in Siqian_Data_Preprocessing, IDs and labels must be aligned in every eligible session
    assert (cn == cln)[eligible].all(), "candidate IDs and labels are misaligned in an eligible session"
    c_local = np.repeat(local, cn)
    lab = lab[eligible[l_local]]
    stats["sessions"] += n
    stats["eligible"] += int(eligible.sum())
    stats["all_negative"] += int((n_pos == 0).sum())
    stats["all_positive"] += int((n_neg == 0).sum())

    keep_c = eligible[c_local]
    cand = pd.DataFrame({
        "session_id": (sid0 + c_local[keep_c]).astype(np.int32),
        "wid": to_int64(cw[keep_c], "candidate product ID"),
        "label": lab.astype(np.int8),
        "position": within_pos(cn)[keep_c].astype(np.int16),
    })

    # ---- history: parse, align, elapsed days (Siqian_Data_Preprocessing convention) ----
    hw, hwn = split_flat(df["history_wid_list"], "_")
    ht, htn = split_flat(df["history_type_list"], "_")
    hq, hqn = split_flat(df["history_qry_list"], "_")
    tm, tmn = split_flat(df["history_time_list"], "_")
    t_all = to_int64(tm, "time gap")
    t_pos = within_pos(tmn)
    gaps = t_all[t_pos >= 1]                          # entries 1..n of each row
    g_len = np.maximum(tmn - 1, 0)
    rev = np.cumsum(gaps[::-1])[::-1]                 # sum of this and all later gaps (across rows)
    after = np.append(rev, 0)[np.cumsum(g_len)]       # part of that sum belonging to later rows
    days_all = (rev - np.repeat(after, g_len)) / 86400.0
    has_t = tmn > 0
    last_gap = t_all[(np.cumsum(tmn) - 1)[has_t]]
    stats["rows_with_times"] += int(eligible[has_t].sum())
    stats["negative_last_gap"] += int((last_gap < 0)[eligible[has_t]].sum())

    n_act = np.minimum(np.minimum(hwn, htn), g_len)   # Siqian_Data_Preprocessing zips wid, type and days
    h_wid = to_int64(select_first(hw, hwn, n_act), "history product ID")
    h_type = pd.Series(select_first(ht, htn, n_act)).map(TYPE_CODE).fillna(-1).to_numpy(np.int8)
    h_days = select_first(days_all, g_len, n_act)
    # past queries: pad with '-1' where the query list is shorter
    out_start = np.cumsum(n_act) - n_act
    h_qry = np.full(int(n_act.sum()), "-1", dtype=object)
    q_pos = within_pos(hqn)
    q_ok = q_pos < np.repeat(n_act, hqn)
    h_qry[np.repeat(out_start, hqn)[q_ok] + q_pos[q_ok]] = hq[q_ok]
    h_local = np.repeat(local, n_act)

    # cleaning (Siqian_Data_Preprocessing): remove carts older than 90 days; then keep eligible sessions
    keep_h = eligible[h_local] & ~old_cart
    stats["carts_removed"] += int((old_cart & eligible[h_local]).sum())
    is_click = h_type[keep_h] == TYPE_CODE["CLICK"]
    stats["clicks"] += int(is_click.sum())
    stats["clicks_within_7d"] += int((h_days[keep_h][is_click] < 7).sum())
    hist = pd.DataFrame({
        "session_id": (sid0 + h_local[keep_h]).astype(np.int32),
        "wid": h_wid[keep_h],
        "type": h_type[keep_h],
        "days": h_days[keep_h].astype(np.float32),
        "hqry": h_qry[keep_h],
    })
    sess = pd.DataFrame({
        "session_id": (sid0 + local[eligible]).astype(np.int32),
        "query": df["query"].fillna("").to_numpy(dtype=object)[eligible],
        "slate_size": cn[eligible].astype(np.int32),
    })
    return cand, hist, sess, stats

In [5]:
def stage_ok():
    # Load the saved session chunks and check that they are complete and consistent.
    if not restore("stage", "sessions.parquet", "step1_stats.json"):
        print("Session chunks not in the backup - rebuilding from the raw user file.")
        return False
    st = json.loads((LOCAL_DIR / "step1_stats.json").read_text())
    n = int(st["chunks"])
    files_ok = all((STAGE_DIR / f"{k}_{i:03d}.parquet").exists() for i in range(n) for k in ["cand", "hist"])
    rows = sum(pq.ParquetFile(STAGE_DIR / f"cand_{i:03d}.parquet").metadata.num_rows for i in range(n)) if files_ok else -1
    if files_ok and rows == st["candidate_rows"]:
        print(f"Reusing the saved session chunks: {n} chunks, {rows:,} candidate rows.")
        return True
    print("Backed-up session chunks are incomplete - rebuilding from the raw user file.")
    shutil.rmtree(STAGE_DIR, ignore_errors=True); STAGE_DIR.mkdir(parents=True)
    return False


def write_chunk(chunk_id, cand, hist, rng):
    # Shuffle candidates within each session (display order is not kept as row order) and write both files.
    order = np.lexsort((rng.random(len(cand)), cand["session_id"].to_numpy()))
    cand = cand.iloc[order].reset_index(drop=True)
    cand.to_parquet(STAGE_DIR / f"cand_{chunk_id:03d}.parquet", index=False)
    hist.to_parquet(STAGE_DIR / f"hist_{chunk_id:03d}.parquet", index=False)
    return np.unique(cand["wid"].to_numpy()), np.unique(hist["wid"].to_numpy())


if stage_ok():
    print("Step 1 already completed - skipping.")
else:
    rng = np.random.default_rng(SEED)
    totals = Counter()
    pend_c, pend_h, pend_s = [], [], []
    sess_rows, needed = [], []
    chunk_id, sid0 = 0, 0
    reader = pd.read_csv(USER_TXT, sep="\t", dtype=str, chunksize=READ_ROWS)
    for block in reader:
        cand, hist, sess, st = parse_block(block, sid0)
        sid0 += len(block)
        totals.update(st)
        pend_c.append(cand); pend_h.append(hist); pend_s.append(sess)
        n_pending = sum(len(s) for s in pend_s)
        while n_pending >= SESSIONS_PER_CHUNK:
            s_all = pd.concat(pend_s, ignore_index=True)
            c_all = pd.concat(pend_c, ignore_index=True)
            h_all = pd.concat(pend_h, ignore_index=True)
            cut = s_all["session_id"].iloc[SESSIONS_PER_CHUNK - 1]
            s_out = s_all[s_all["session_id"] <= cut].assign(chunk=chunk_id)
            uc, uh = write_chunk(chunk_id, c_all[c_all["session_id"] <= cut], h_all[h_all["session_id"] <= cut], rng)
            sess_rows.append(s_out); needed += [uc, uh]
            pend_s = [s_all[s_all["session_id"] > cut]]
            pend_c = [c_all[c_all["session_id"] > cut]]
            pend_h = [h_all[h_all["session_id"] > cut]]
            chunk_id += 1
            n_pending = len(pend_s[0])
        print(f"read {sid0:,} rows -> {chunk_id} chunks written", end="\r")
        del block, cand, hist, sess
    # final partial chunk
    s_all = pd.concat(pend_s, ignore_index=True)
    if len(s_all):
        uc, uh = write_chunk(chunk_id, pd.concat(pend_c, ignore_index=True), pd.concat(pend_h, ignore_index=True), rng)
        sess_rows.append(s_all.assign(chunk=chunk_id)); needed += [uc, uh]
        chunk_id += 1
    sessions = pd.concat(sess_rows, ignore_index=True)
    sessions.to_parquet(LOCAL_DIR / "sessions.parquet", index=False)
    np.save(ARR_DIR / "needed_wids.npy", np.unique(np.concatenate(needed)))
    totals["chunks"] = chunk_id
    totals["candidate_rows"] = int(sessions["slate_size"].sum())
    (LOCAL_DIR / "step1_stats.json").write_text(json.dumps({k: int(v) for k, v in totals.items()}))
    del pend_c, pend_h, pend_s, sess_rows, needed, s_all, sessions
    gc.collect()
    mark_done("step1")
    backup("stage", "sessions.parquet", "arrays/needed_wids.npy", "step1_stats.json")

stats1 = json.loads((LOCAL_DIR / "step1_stats.json").read_text())
sessions = pd.read_parquet(LOCAL_DIR / "sessions.parquet")
N_CHUNKS = int(stats1["chunks"])
print(f"Sessions in file:            {stats1['sessions']:,} (expected {EXPECTED['sessions']:,})")
print(f"Eligible sessions:           {stats1['eligible']:,} (expected {EXPECTED['eligible']:,})")
print(f"  excluded all-negative:     {stats1['all_negative']:,};  all-positive: {stats1['all_positive']:,}")
print(f"Candidate rows:              {stats1['candidate_rows']:,} (expected {EXPECTED['candidate_rows']:,})")
print(f"Chunks of {SESSIONS_PER_CHUNK} sessions:      {N_CHUNKS}")
print(f"Cart actions removed (>90d): {stats1['carts_removed']:,} (expected {EXPECTED['carts_removed']:,})")
print(f"Check - negative final gaps: {stats1['negative_last_gap'] / max(stats1['rows_with_times'], 1):.4%} of sessions (should be 0)")
print(f"Check - clicks within 7 days: {stats1['clicks_within_7d'] / max(stats1['clicks'], 1):.2%} (should be close to 100%)")
ram("after step 1")

Reusing the saved session chunks: 86 chunks, 15,348,049 candidate rows.
Step 1 already completed - skipping.
Sessions in file:            173,831 (expected 173,831)
Eligible sessions:           171,175 (expected 171,175)
  excluded all-negative:     2,103;  all-positive: 553
Candidate rows:              15,348,049 (expected 15,348,049)
Chunks of 2000 sessions:      86
Cart actions removed (>90d): 1,177 (expected 1,177)
Check - negative final gaps: 0.0000% of sessions (should be 0)
Check - clicks within 7 days: 100.00% (should be close to 100%)
[RAM] after step 1: 0.31 GB in use


In [6]:
from sklearn.model_selection import train_test_split

ids = pd.Series(sessions["session_id"].to_numpy(), name="session_id")
train_ids, temp_ids = train_test_split(ids, test_size=0.20, random_state=SEED, shuffle=True)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.50, random_state=SEED, shuffle=True)
split_map = pd.Series("train", index=ids.to_numpy())
split_map.loc[val_ids.to_numpy()] = "val"
split_map.loc[test_ids.to_numpy()] = "test"
sessions["split"] = split_map.reindex(sessions["session_id"].to_numpy()).to_numpy()
sessions.to_parquet(LOCAL_DIR / "sessions.parquet", index=False)
backup("sessions.parquet")
print(sessions["split"].value_counts())
del ids, train_ids, temp_ids, val_ids, test_ids

split
train    136940
test      17118
val       17117
Name: count, dtype: int64


## Step 2 — Product tables (attributes and title TF-IDF)

This step loads the product tables saved by `Part3_Feature_Engineering.ipynb` and checks that their sizes agree with one another:
- the lookup arrays have equal length;
- the attribute table has one row per product plus the empty row for products without metadata;
- the TF-IDF row pointers have one entry per product plus two;
- the index and data files each hold exactly the number of entries the row pointers announce;
- the vocabulary has the size recorded in `prod_meta.json`.

Only if a check fails are the tables rebuilt from the product metadata with the same code as that notebook. That code keeps the products occurring as candidates or in histories, stores brand, shop and four category levels as integers, and builds unit-length TF-IDF title vectors from terms in at least 5 titles. A rebuild also recreates the title text cache, which Step 3 needs only if term2vec has to be retrained.

The helper cell that follows opens the tables memory-mapped, so the 11.5 million title vectors never have to be loaded into RAM at once.

In [7]:
def product_tables_ok():
    # Load the saved product tables and check that their sizes agree with one another.
    if not restore(*[f"arrays/{n}" for n in PRODUCT_FILES]):
        print("Product tables not (fully) in the backup - rebuilding.")
        return False
    try:
        meta = json.loads((ARR_DIR / "prod_meta.json").read_text())
        indptr = np.load(ARR_DIR / "title_indptr.npy", mmap_mode="r"); nnz = int(indptr[-1])
        checks = {
            "lookup arrays equal length": len(np.load(ARR_DIR / "lookup_wid.npy", mmap_mode="r")) == len(np.load(ARR_DIR / "lookup_row.npy", mmap_mode="r")),
            "attribute table rows": np.load(ARR_DIR / "prod_attr.npy", mmap_mode="r").shape == (meta["n_prod"] + 1, 6),
            "TF-IDF row pointers": len(indptr) == meta["n_prod"] + 2,
            "TF-IDF index file complete": (ARR_DIR / "title_indices.bin").stat().st_size == 4 * nnz,
            "TF-IDF data file complete": (ARR_DIR / "title_data.bin").stat().st_size == 4 * nnz,
            "vocabulary size": len(pd.read_parquet(ARR_DIR / "title_vocab.parquet")) == meta["vocab"],
        }
    except Exception as e:
        checks = {f"readable ({type(e).__name__})": False}
    for k, v in checks.items():
        print(f"  {k:30s} {'OK' if v else 'FAILED'}")
    if all(checks.values()):
        print("Reusing the saved product tables.")
        return True
    print("Product tables inconsistent - rebuilding.")
    for n in PRODUCT_FILES:
        (ARR_DIR / n).unlink(missing_ok=True)
    return False


def ensure_needed_wids():
    # The product IDs occurring as candidates or in histories (written by Step 1; recomputed if absent).
    f = ARR_DIR / "needed_wids.npy"
    if not f.exists() and not restore("arrays/needed_wids.npy"):
        parts = []
        for i in range(N_CHUNKS):
            for k in ["cand", "hist"]:
                parts.append(np.unique(pd.read_parquet(STAGE_DIR / f"{k}_{i:03d}.parquet", columns=["wid"])["wid"].to_numpy()))
        np.save(f, np.unique(np.concatenate(parts)))
    return np.load(f)


def product_chunks(columns):
    # Yield the product metadata in chunks, from the Parquet cache if present, else from the text file.
    cached = sorted(PRODUCT_CACHE_DIR.glob("*.parquet")) if PRODUCT_CACHE_DIR.exists() else []
    if cached:
        for f in cached:
            yield pd.read_parquet(f, columns=columns)
    else:
        for chunk in pd.read_csv(PRODUCT_TXT, sep="\t", dtype=str, usecols=columns, chunksize=500_000):
            yield chunk


def isin_sorted(values, sorted_ref):
    i = np.searchsorted(sorted_ref, values)
    i = np.minimum(i, len(sorted_ref) - 1)
    return sorted_ref[i] == values


if product_tables_ok():
    print("Step 2 already completed - skipping.")
else:
    needed = ensure_needed_wids()
    wid_parts, attr_parts, n_rows, n_bad = [], [], 0, 0
    for i, ch in enumerate(product_chunks(["wid", "name"] + ATTR_SOURCE)):
        w = pd.to_numeric(ch["wid"], errors="coerce")
        n_bad += int(w.isna().sum())
        keep = w.notna().to_numpy()
        keep[keep] = isin_sorted(w[keep].to_numpy(np.int64), needed)
        ch = ch[keep]
        wid_parts.append(w[keep].to_numpy(np.int64))
        a = np.column_stack([pd.to_numeric(ch[c], errors="coerce").fillna(-1).to_numpy(np.int64) for c in ATTR_SOURCE])
        attr_parts.append(a)
        ch[["name"]].fillna("").to_parquet(TITLE_DIR / f"titles_{i:03d}.parquet", index=False)
        n_rows += len(ch)
        print(f"product chunk {i}: kept {len(ch):,} rows (total {n_rows:,})", end="\r")
    prod_wid = np.concatenate(wid_parts); prod_attr = np.vstack(attr_parts + [np.full((1, 6), -1, np.int64)])
    del wid_parts, attr_parts
    N_PROD = len(prod_wid)
    # lookup: sorted unique IDs -> first row
    order = np.argsort(prod_wid, kind="stable")
    sw = prod_wid[order]
    first = np.ones(len(sw), bool); first[1:] = sw[1:] != sw[:-1]
    np.save(ARR_DIR / "lookup_wid.npy", sw[first]); np.save(ARR_DIR / "lookup_row.npy", order[first].astype(np.int64))
    np.save(ARR_DIR / "prod_attr.npy", prod_attr)
    meta = {"n_prod": int(N_PROD), "duplicates": int((~first).sum()), "non_integer_wid": n_bad,
            "needed": int(len(needed)), "needed_missing": int(len(needed) - first.sum())}
    del prod_wid, prod_attr, order, sw, first, needed
    gc.collect()

    # ---- title TF-IDF: pass 1 counts document frequencies, pass 2 writes the matrix ----
    title_files = sorted(TITLE_DIR.glob("titles_*.parquet"))
    df_count = Counter()
    for f in title_files:
        names = pd.read_parquet(f)["name"].to_numpy(dtype=object)
        df_count.update(chain.from_iterable(set(x.split(TERM_SEP)) - {""} for x in names))
    vocab = sorted(t for t, c in df_count.items() if c >= TITLE_MIN_DF)
    term_df = np.array([df_count[t] for t in vocab], dtype=np.float64)
    idf = np.log((1 + N_PROD) / (1 + term_df)) + 1.0
    vocab_index = pd.Index(vocab)
    pd.DataFrame({"term": vocab}).to_parquet(ARR_DIR / "title_vocab.parquet", index=False)
    del df_count, term_df

    indptr, offset = [np.zeros(1, np.int64)], 0
    with open(ARR_DIR / "title_indices.bin", "wb") as fi, open(ARR_DIR / "title_data.bin", "wb") as fd:
        for f in title_files:
            names = pd.read_parquet(f)["name"].to_numpy(dtype=object)
            toks, lens = split_flat(pd.Series(names), TERM_SEP)
            rows = np.repeat(np.arange(len(names)), lens)
            cols = vocab_index.get_indexer(toks)
            ok = cols >= 0
            m = sp.csr_matrix((np.ones(ok.sum(), np.float32), (rows[ok], cols[ok])), shape=(len(names), len(vocab)))
            m.sum_duplicates()                               # term frequencies (duplicates summed)
            m.data *= idf[m.indices].astype(np.float32)      # tf x idf
            m = normalize(m, norm="l2", copy=False)          # unit length: dot product = cosine
            m.indices.astype(np.int32).tofile(fi)
            m.data.astype(np.float32).tofile(fd)
            indptr.append(offset + m.indptr[1:].astype(np.int64))
            offset += int(m.nnz)
            del names, toks, lens, rows, cols, m
    indptr = np.concatenate(indptr)
    indptr = np.append(indptr, indptr[-1])                   # the extra empty row for unknown products
    np.save(ARR_DIR / "title_indptr.npy", indptr)
    meta.update({"vocab": len(vocab), "title_nnz": int(indptr[-1])})
    (ARR_DIR / "prod_meta.json").write_text(json.dumps(meta))
    del vocab, vocab_index, idf, indptr
    gc.collect()
    mark_done("step3")
    backup(*[f"arrays/{n}" for n in PRODUCT_FILES])

PROD_META = json.loads((ARR_DIR / "prod_meta.json").read_text())
print(PROD_META)
ram("after step 2")

  lookup arrays equal length     OK
  attribute table rows           OK
  TF-IDF row pointers            OK
  TF-IDF index file complete     OK
  TF-IDF data file complete      OK
  vocabulary size                OK
Reusing the saved product tables.
Step 2 already completed - skipping.
{'n_prod': 11500314, 'duplicates': 0, 'non_integer_wid': 0, 'needed': 12223304, 'needed_missing': 722990, 'vocab': 368021, 'title_nnz': 268204961}
[RAM] after step 2: 0.36 GB in use


In [8]:
LOOKUP_WID = np.load(ARR_DIR / "lookup_wid.npy", mmap_mode="r")
LOOKUP_ROW = np.load(ARR_DIR / "lookup_row.npy", mmap_mode="r")
PROD_ATTR = np.load(ARR_DIR / "prod_attr.npy", mmap_mode="r")
EMPTY_ROW = PROD_META["n_prod"]
TITLE_VOCAB = pd.read_parquet(ARR_DIR / "title_vocab.parquet")["term"]
TITLE_INDEX = pd.Index(TITLE_VOCAB)


def prod_rows(wids):
    # Row of each product ID in the product tables (EMPTY_ROW if the product has no metadata).
    wids = np.asarray(wids, dtype=np.int64)
    i = np.searchsorted(LOOKUP_WID, wids)
    i_c = np.minimum(i, len(LOOKUP_WID) - 1)
    found = np.asarray(LOOKUP_WID[i_c]) == wids
    return np.where(found, np.asarray(LOOKUP_ROW[i_c]), EMPTY_ROW)


def open_title_matrix():
    indptr = np.load(ARR_DIR / "title_indptr.npy", mmap_mode="r")
    nnz = int(indptr[-1])
    indices = np.memmap(ARR_DIR / "title_indices.bin", dtype=np.int32, mode="r", shape=(nnz,))
    data = np.memmap(ARR_DIR / "title_data.bin", dtype=np.float32, mode="r", shape=(nnz,))
    return sp.csr_matrix((data, indices, np.asarray(indptr)), shape=(len(indptr) - 1, len(TITLE_VOCAB)))


TITLES = open_title_matrix()
print("title matrix:", TITLES.shape, "non-zeros:", TITLES.nnz)

title matrix: (11500315, 368021) non-zeros: 268204961


## Step 3 — Embeddings (item2vec and term2vec)

This step loads the embeddings saved by `Part3_Feature_Engineering.ipynb` and checks them:
- the item2vec keys (sorted product IDs) must match the vectors in number and be sorted;
- the term2vec matrix must have one row per TF-IDF vocabulary term.

Only if a check fails is the model retrained with the same settings: word2vec skip-gram, 64 dimensions, window 5, 5 epochs. Item2vec is trained on the cleaned histories of all eligible sessions, products seen at least 5 times. Term2vec is trained on the product titles from the title cache. A retrained model would not be bit-for-bit identical to the saved one, because gensim trains on several threads, so the saved vectors are reused whenever they pass the checks. Both are reused as they are for the window features, because they describe products, not a user's history.

In [11]:
from gensim.models import Word2Vec


class HistoryCorpus:
    # One sentence (list of product-ID strings) per session, streamed from disk.
    def __iter__(self):
        for i in range(N_CHUNKS):
            h = pd.read_parquet(STAGE_DIR / f"hist_{i:03d}.parquet", columns=["session_id", "wid"])
            sid = h["session_id"].to_numpy()
            toks = h["wid"].to_numpy().astype(str)
            cuts = np.flatnonzero(np.diff(sid)) + 1
            for sent in np.split(toks, cuts):
                if len(sent):
                    yield sent.tolist()


class TitleCorpus:
    # One sentence (list of term IDs) per product title, streamed from disk.
    def __iter__(self):
        for f in sorted(TITLE_DIR.glob("titles_*.parquet")):
            for name in pd.read_parquet(f)["name"].to_numpy(dtype=object):
                if name:
                    yield name.split(TERM_SEP)


def i2v_ok():
    if not restore(*[f"arrays/{n}" for n in I2V_FILES]):
        return False
    try:
        k = np.load(ARR_DIR / "i2v_keys.npy", mmap_mode="r"); v = np.load(ARR_DIR / "i2v_vecs.npy", mmap_mode="r")
        return len(k) == v.shape[0] and v.shape[1] == EMB_DIM and bool(np.all(np.diff(k[:100_000]) > 0))
    except Exception:
        return False


def t2v_ok():
    if not restore(*[f"arrays/{n}" for n in T2V_FILES]):
        return False
    try:
        return np.load(ARR_DIR / "t2v_term_vecs.npy", mmap_mode="r").shape == (len(TITLE_VOCAB), EMB_DIM)
    except Exception:
        return False


def ensure_title_cache():
    # Title texts of the needed products, one Parquet file per product chunk (as written by Step 2).
    if any(TITLE_DIR.glob("titles_*.parquet")) or restore("titles"):
        return
    print("Title cache not available - re-extracting titles from the product file.")
    needed = ensure_needed_wids()
    for i, ch in enumerate(product_chunks(["wid", "name"])):
        w = pd.to_numeric(ch["wid"], errors="coerce")
        keep = w.notna().to_numpy()
        keep[keep] = isin_sorted(w[keep].to_numpy(np.int64), needed)
        ch[keep][["name"]].fillna("").to_parquet(TITLE_DIR / f"titles_{i:03d}.parquet", index=False)


workers = max(1, os.cpu_count() or 1)
if i2v_ok():
    print("Reusing the saved item2vec vectors.")
else:
    print("Item2vec vectors missing or inconsistent - retraining with the same settings.")
    t0 = time.time()
    i2v = Word2Vec(HistoryCorpus(), vector_size=EMB_DIM, window=EMB_WINDOW, min_count=I2V_MIN_COUNT, sg=1,
                   epochs=EMB_EPOCHS, workers=workers, seed=SEED)
    keys = np.array(i2v.wv.index_to_key, dtype=np.int64)
    vecs = normalize(i2v.wv.vectors).astype(np.float32)
    order = np.argsort(keys)
    np.save(ARR_DIR / "i2v_keys.npy", keys[order]); np.save(ARR_DIR / "i2v_vecs.npy", vecs[order])
    print(f"item2vec: {len(keys):,} products with vectors ({time.time() - t0:.0f}s)")
    del i2v, keys, vecs, order; gc.collect()
    backup(*[f"arrays/{n}" for n in I2V_FILES])

if t2v_ok():
    print("Reusing the saved term2vec vectors.")
else:
    print("Term2vec vectors missing or inconsistent - retraining with the same settings.")
    ensure_title_cache()
    t0 = time.time()
    t2v = Word2Vec(TitleCorpus(), vector_size=EMB_DIM, window=EMB_WINDOW, min_count=TITLE_MIN_DF, sg=1,
                   epochs=EMB_EPOCHS, workers=workers, seed=SEED)
    E = np.zeros((len(TITLE_VOCAB), EMB_DIM), np.float32)
    k2i = t2v.wv.key_to_index
    pos = np.array([k2i.get(t, -1) for t in TITLE_VOCAB])
    E[pos >= 0] = t2v.wv.vectors[pos[pos >= 0]]
    np.save(ARR_DIR / "t2v_term_vecs.npy", E)
    print(f"term2vec: {(pos >= 0).sum():,} of {len(TITLE_VOCAB):,} title terms with vectors ({time.time() - t0:.0f}s)")
    del t2v, E, k2i, pos; gc.collect()
    backup(*[f"arrays/{n}" for n in T2V_FILES])
shutil.rmtree(TITLE_DIR, ignore_errors=True)    # the title texts are not needed beyond this point

I2V_KEYS = np.load(ARR_DIR / "i2v_keys.npy", mmap_mode="r")
I2V_VECS = np.load(ARR_DIR / "i2v_vecs.npy", mmap_mode="r")
T2V_TERMS = np.load(ARR_DIR / "t2v_term_vecs.npy", mmap_mode="r")


def i2v_rows(wids):
    # Index of each product's item2vec vector, or -1 if it has none.
    wids = np.asarray(wids, dtype=np.int64)
    if len(I2V_KEYS) == 0:
        return np.full(len(wids), -1)
    i = np.minimum(np.searchsorted(I2V_KEYS, wids), len(I2V_KEYS) - 1)
    return np.where(np.asarray(I2V_KEYS[i]) == wids, i, -1)


print("item vectors:", I2V_VECS.shape, " term vectors:", T2V_TERMS.shape)
ram("after step 3")

Reusing the saved item2vec vectors.
Reusing the saved term2vec vectors.
item vectors: (859412, 64)  term vectors: (368021, 64)
[RAM] after step 3: 0.47 GB in use


## Step 4 — Training-set priors (family G)

This step loads the saved priors files when there is one per chunk with the same number of rows as the chunk's candidates. Otherwise it recomputes them with the same code as `Part3_Feature_Engineering.ipynb`: smoothed engagement rates (pos + 20·p₀) / (n + 20) from training sessions only, out-of-fold for training rows (5 folds, so no training row's own label enters its own feature), and log(1 + training impressions). The priors are computed from the candidate labels, not from the history, so they are the same for every history window and are reused unchanged in Step 6.

In [12]:
def priors_ok():
    # Priors are reusable only if every chunk has a priors file with as many rows as its candidates.
    for i in range(N_CHUNKS):
        f = STAGE_DIR / f"priors_{i:03d}.parquet"
        if not f.exists() and not restore(f"stage/priors_{i:03d}.parquet"):
            print("Priors missing - recomputing."); return False
        if pq.ParquetFile(f).metadata.num_rows != pq.ParquetFile(STAGE_DIR / f"cand_{i:03d}.parquet").metadata.num_rows:
            print("Priors do not match the session chunks - recomputing."); return False
    print("Reusing the saved training-set priors.")
    return True


if priors_ok():
    print("Step 4 already completed - skipping.")
else:
    parts = []
    for i in range(N_CHUNKS):
        c = pd.read_parquet(STAGE_DIR / f"cand_{i:03d}.parquet", columns=["session_id", "wid", "label"])
        a = PROD_ATTR[prod_rows(c["wid"].to_numpy())]
        c["brand"] = a[:, BRAND]; c["cate3"] = a[:, CATE3]
        c["chunk"] = np.int16(i)
        parts.append(c)
    cand_all = pd.concat(parts, ignore_index=True); del parts
    split_of = sessions.set_index("session_id")["split"]
    cand_all["split"] = pd.Categorical(split_of.reindex(cand_all["session_id"].to_numpy()).to_numpy())
    cand_all["pos"] = (cand_all["label"] > 0).astype(np.int8)

    # folds over training sessions
    tr_sess = sessions.loc[sessions["split"] == "train", "session_id"].to_numpy()
    fold_of = pd.Series(np.random.default_rng(SEED).permutation(len(tr_sess)) % N_FOLDS, index=tr_sess)
    is_tr = (cand_all["split"] == "train").to_numpy()
    cand_all["fold"] = np.int8(-1)
    cand_all.loc[is_tr, "fold"] = fold_of.reindex(cand_all.loc[is_tr, "session_id"].to_numpy()).to_numpy()
    tr = cand_all[is_tr]
    p0 = tr["pos"].mean()
    print(f"training rows: {len(tr):,}; positive rate p0 = {p0:.4f}")

    for key, col in [("wid", "item_ctr_train"), ("brand", "brand_ctr_train"), ("cate3", "cate3_ctr_train")]:
        full = tr.groupby(key)["pos"].agg(["size", "sum"])
        by_fold = tr.groupby([key, "fold"])["pos"].agg(["size", "sum"])
        keys = cand_all[key].to_numpy()
        n_full = full["size"].reindex(keys).fillna(0).to_numpy()
        p_full = full["sum"].reindex(keys).fillna(0).to_numpy()
        idx = pd.MultiIndex.from_arrays([keys[is_tr], cand_all.loc[is_tr, "fold"].to_numpy()])
        n_full[is_tr] -= by_fold["size"].reindex(idx).fillna(0).to_numpy()   # leave own fold out
        p_full[is_tr] -= by_fold["sum"].reindex(idx).fillna(0).to_numpy()
        rate = (p_full + PRIOR_ALPHA * p0) / (n_full + PRIOR_ALPHA)
        if key != "wid":
            rate = np.where(keys == -1, np.nan, rate)
        cand_all[col] = rate.astype(np.float32)
        if key == "wid":
            cand_all["item_impr_train"] = np.log1p(full["size"].reindex(keys).fillna(0).to_numpy()).astype(np.float32)
        del full, by_fold, keys, n_full, p_full, idx, rate
    for i, g in cand_all.groupby("chunk", sort=True):
        g[["session_id", "wid"] + FAMILIES["G"]].to_parquet(STAGE_DIR / f"priors_{i:03d}.parquet", index=False)
    del cand_all, tr, fold_of
    gc.collect()
    mark_done("step5")
    backup(*[f"stage/priors_{i:03d}.parquet" for i in range(N_CHUNKS)])
ram("after step 4")

Reusing the saved training-set priors.
Step 4 already completed - skipping.
[RAM] after step 4: 0.47 GB in use


## Step 5 — Feature computation

The next cell defines `build_chunk_features`, the same function as in `Part3_Feature_Engineering.ipynb`. For one chunk of 2,000 sessions, it computes all 90 features from the chunk's candidates, histories and priors. In short:
- every past action carries the weight w = type weight × exp(−elapsed days / 30), with CLICK 1, FLW 2, CART 3 and ORD 5, so recent and stronger actions count for more;
- the families are computed in turn: context, A, B, C, D, E, F and G; family H comes last, because it is defined relative to the other candidates of the same slate;
- a missing value (`NaN`) means "no evidence" and is not replaced by zero.

Because the function reads the history from `STAGE_DIR`, Step 6 can point `STAGE_DIR` at a folder of cut-down histories and run the identical code, which is what makes the window features directly comparable with the full-year ones.

In [13]:
def group_bounds(sorted_local, n_groups):
    # Start and end row of each group in an array sorted by group.
    starts = np.searchsorted(sorted_local, np.arange(n_groups), side="left")
    ends = np.searchsorted(sorted_local, np.arange(n_groups), side="right")
    return starts, ends


def rowdot(A, B):
    # Row-wise dot product of two sparse matrices with the same shape.
    return np.asarray(A.multiply(B).sum(axis=1)).ravel()


def nan_if(values, mask):
    out = np.asarray(values, dtype=np.float64).copy()
    out[mask] = np.nan
    return out


def keyed_merge(c_keys, h_keys, h_values, how="sum"):
    # For each candidate key (session, value), aggregate h_values over history rows with the same key.
    # Keys are 2-column int64 arrays; returns NaN where no history row matches.
    h = pd.DataFrame({"s": h_keys[0], "v": h_keys[1], "x": h_values})
    agg = h.groupby(["s", "v"])["x"].agg(how)
    idx = pd.MultiIndex.from_arrays([c_keys[0], c_keys[1]])
    return agg.reindex(idx).to_numpy(dtype=np.float64)


def build_chunk_features(i):
    cand = pd.read_parquet(STAGE_DIR / f"cand_{i:03d}.parquet")
    hist = pd.read_parquet(STAGE_DIR / f"hist_{i:03d}.parquet")
    pri = pd.read_parquet(STAGE_DIR / f"priors_{i:03d}.parquet")
    assert (pri["session_id"].to_numpy() == cand["session_id"].to_numpy()).all()
    assert (pri["wid"].to_numpy() == cand["wid"].to_numpy()).all()
    sess = sessions[sessions["chunk"] == i].sort_values("session_id")
    sids = sess["session_id"].to_numpy()
    nS = len(sids)
    c_loc = np.searchsorted(sids, cand["session_id"].to_numpy())
    h_loc = np.searchsorted(sids, hist["session_id"].to_numpy())
    nC, nH = len(cand), len(hist)
    F = {}

    c_wid, h_wid = cand["wid"].to_numpy(), hist["wid"].to_numpy()
    c_row, h_row = prod_rows(c_wid), prod_rows(h_wid)
    c_attr, h_attr = np.asarray(PROD_ATTR[c_row]), np.asarray(PROD_ATTR[h_row])
    typ = hist["type"].to_numpy().astype(np.int64)
    days = hist["days"].to_numpy().astype(np.float64)
    w = np.where(typ >= 0, TYPE_WEIGHT[np.clip(typ, 0, 3)], 1.0) * np.exp(-days / TAU_DAYS)
    tot_w = np.bincount(h_loc, weights=w, minlength=nS)
    h_len = np.bincount(h_loc, minlength=nS)
    h_start, h_end = group_bounds(h_loc, nS)
    c_start, c_end = group_bounds(c_loc, nS)
    has_h = h_len > 0
    last_idx = np.where(has_h, h_end - 1, 0)

    # ---------------- context (session level, broadcast to candidates) ----------------
    ctx = {}
    for t, tname in enumerate(TYPE_NAMES):
        for (a, b), wname in zip([(0, 7), (7, 90), (90, 365)], ["0_7d", "7_90d", "90_365d"]):
            m = (typ == t) & (days >= a) & (days < b)
            ctx[f"{tname}_count_{wname}"] = np.bincount(h_loc[m], minlength=nS)
    ctx["history_len_clean"] = h_len
    dser = pd.Series(days).groupby(h_loc)
    ctx["history_span_days"] = dser.max().reindex(range(nS)).to_numpy()
    ctx["days_since_last_interaction"] = dser.min().reindex(range(nS)).to_numpy()
    for t, tname in enumerate(TYPE_NAMES):
        ctx[f"share_{tname}"] = nan_if(np.bincount(h_loc[typ == t], minlength=nS) / np.maximum(h_len, 1), ~has_h)
    c3 = h_attr[:, CATE3]
    m3 = c3 != -1
    cc = pd.DataFrame({"s": h_loc[m3], "v": c3[m3]}).value_counts()
    p = cc / cc.groupby(level=0).transform("sum")
    ent = (-(p * np.log(p))).groupby(level=0).sum()
    ctx["cate3_entropy"] = nan_if(ent.reindex(range(nS)).fillna(0).to_numpy(), ~has_h)
    br = h_attr[:, BRAND]
    ctx["n_distinct_brands"] = pd.DataFrame({"s": h_loc[br != -1], "v": br[br != -1]}).drop_duplicates()["s"] \
        .value_counts().reindex(range(nS)).fillna(0).to_numpy()
    qryless = (hist["hqry"].to_numpy(dtype=object) == "-1")
    ctx["qryless_share"] = nan_if(np.bincount(h_loc[qryless], minlength=nS) / np.maximum(h_len, 1), ~has_h)
    same_next = np.zeros(nH, bool); same_next[:-1] = h_loc[:-1] == h_loc[1:]
    gap_next = np.zeros(nH); gap_next[:-1] = days[:-1] - days[1:]
    brk = same_next & (gap_next >= BURST_GAP_DAYS)
    hpos = np.arange(nH) - h_start[h_loc]
    last_brk = np.full(nS, -1); np.maximum.at(last_brk, h_loc[brk], hpos[brk])
    ctx["burst_len"] = h_len - (last_brk + 1)
    for k in FAMILIES["context"]:
        F[k] = np.asarray(ctx[k], dtype=np.float64)[c_loc]

    # ---------------- A. exact product ----------------
    defs = [("exact_wid_click_0_7d", 0, 0, 7), ("exact_wid_cart_0_7d", 1, 0, 7), ("exact_wid_ord_0_7d", 2, 0, 7),
            ("exact_wid_cart_7_90d", 1, 7, 90), ("exact_wid_ord_7_90d", 2, 7, 90), ("exact_wid_ord_90_365d", 2, 90, 365)]
    hdf = pd.DataFrame({"s": h_loc, "v": h_wid, "d": days})
    for name, t, a, b in defs:
        hdf[name] = ((typ == t) & (days >= a) & (days < b)).astype(np.int32)
    g = hdf.groupby(["s", "v"]).agg({**{n: "sum" for n, *_ in defs}, "d": "min"})
    idx = pd.MultiIndex.from_arrays([c_loc, c_wid])
    ga = g.reindex(idx)
    for name, *_ in defs:
        F[name] = ga[name].fillna(0).to_numpy()
    F["exact_wid_last_seen_days"] = ga["d"].to_numpy()
    del hdf, g, ga

    # ---------------- B. attribute affinity ----------------
    is_ord = (typ == TYPE_CODE["ORD"]).astype(np.float64)
    for j, k in enumerate(ATTRS):
        hv, cv = h_attr[:, j], c_attr[:, j]
        hm = hv != -1
        keys_h = (h_loc[hm], hv[hm])
        keys_c = (c_loc, cv)
        aff = keyed_merge(keys_c, keys_h, w[hm], "sum")
        last = keyed_merge(keys_c, keys_h, days[hm], "min")
        ordc = keyed_merge(keys_c, keys_h, is_ord[hm], "sum")
        miss = cv == -1
        F[f"{k}_aff"] = nan_if(aff, miss)
        F[f"{k}_share"] = nan_if(aff / np.maximum(tot_w[c_loc], 1e-12), miss)
        F[f"{k}_last_seen_days"] = nan_if(last, miss)
        F[f"{k}_ord_cnt"] = nan_if(ordc, miss)

    # ---------------- C. title similarity ----------------
    Xc, Xh = TITLES[c_row], TITLES[h_row]
    c_nnz = np.diff(Xc.indptr)
    S = sp.csr_matrix((w, (h_loc, np.arange(nH))), shape=(nS, nH))
    P = normalize(S @ Xh)
    p_nnz = np.diff(P.indptr)
    F["title_cos_profile"] = nan_if(rowdot(P[c_loc], Xc), (c_nnz == 0) | (p_nnz[c_loc] == 0))
    S_ord = sp.csr_matrix((w * is_ord, (h_loc, np.arange(nH))), shape=(nS, nH))
    P_ord = normalize(S_ord @ Xh)
    po_nnz = np.diff(P_ord.indptr)
    F["title_cos_profile_ord"] = nan_if(rowdot(P_ord[c_loc], Xc), (c_nnz == 0) | (po_nnz[c_loc] == 0))
    Xlast = Xh[last_idx]
    l_nnz = np.diff(Xlast.indptr)
    F["last_title_cos"] = nan_if(rowdot(Xlast[c_loc], Xc), (c_nnz == 0) | (l_nnz[c_loc] == 0) | ~has_h[c_loc])
    # overlap of candidate title terms with the user's past query terms
    hq = hist["hqry"].to_numpy(dtype=object)
    qm = ~qryless
    q_toks, q_lens = split_flat(pd.Series(hq[qm]), TERM_SEP)
    q_cols = TITLE_INDEX.get_indexer(q_toks)
    q_rows = np.repeat(h_loc[qm], q_lens)
    ok = q_cols >= 0
    Q = sp.csr_matrix((np.ones(ok.sum(), np.float32), (q_rows[ok], q_cols[ok])), shape=(nS, len(TITLE_VOCAB)))
    Q.data[:] = 1.0                                            # term present (duplicates collapsed)
    Bc = Xc.copy(); Bc.data[:] = 1.0
    F["title_hist_qry_overlap"] = nan_if(rowdot(Q[c_loc], Bc) / np.maximum(c_nnz, 1), c_nnz == 0)

    # ---------------- D. embedding similarity ----------------
    c_vi, h_vi = i2v_rows(c_wid), i2v_rows(h_wid)
    Vc = np.zeros((nC, EMB_DIM), np.float32); Vc[c_vi >= 0] = I2V_VECS[c_vi[c_vi >= 0]]
    hk = h_vi >= 0
    Vh = np.zeros((nH, EMB_DIM), np.float32); Vh[hk] = I2V_VECS[h_vi[hk]]
    n_known = np.bincount(h_loc[hk], minlength=nS)

    def session_sum(mask, M):
        # Sum the rows of M selected by mask within each session (a sparse indicator product, no Python loop).
        A = sp.csr_matrix((np.ones(mask.sum()), (h_loc[mask], np.flatnonzero(mask))), shape=(nS, nH))
        return np.asarray(A @ M)

    mean_v = session_sum(hk, Vh) / np.maximum(n_known, 1)[:, None]
    c_unknown = c_vi < 0
    F["i2v_cos_mean"] = nan_if((Vc * mean_v[c_loc]).sum(1), c_unknown | (n_known[c_loc] == 0))
    cum = np.cumsum(hk)
    within = cum - np.r_[0, cum][h_start][h_loc]              # known vectors up to and including this row
    rank_from_end = n_known[h_loc] - within + hk               # 1 = most recent known vector of the session
    last5 = hk & (rank_from_end <= 5)
    l5 = normalize(session_sum(last5, Vh))
    F["i2v_cos_last5"] = nan_if((Vc * l5[c_loc]).sum(1), c_unknown | (n_known[c_loc] == 0))
    F["last_i2v_cos"] = nan_if((Vc * Vh[last_idx][c_loc]).sum(1), c_unknown | ~has_h[c_loc] | ~hk[last_idx][c_loc])
    Tc = normalize(np.asarray(Xc @ T2V_TERMS)); Th = normalize(np.asarray(Xh @ T2V_TERMS))
    th_ok = np.abs(Th).sum(1) > 0
    tprof = normalize(session_sum(th_ok, Th))
    t_nn = np.abs(tprof).sum(1) > 0
    F["t2v_cos_profile"] = nan_if((Tc * tprof[c_loc]).sum(1), (np.abs(Tc).sum(1) == 0) | ~t_nn[c_loc])

    # maximum similarity with any single history item (loop over sessions)
    tmax = np.full(nC, np.nan); imax = np.full(nC, np.nan)
    for s in range(nS):
        a, b, c0, d0 = c_start[s], c_end[s], h_start[s], h_end[s]
        if b == a or d0 == c0:
            continue
        M = (Xc[a:b] @ Xh[c0:d0].T).toarray()
        tmax[a:b] = M.max(axis=1)
        kh = hk[c0:d0]
        if kh.any():
            imax[a:b] = (Vc[a:b] @ Vh[c0:d0][kh].T).max(axis=1)
    F["title_cos_max"] = nan_if(tmax, c_nnz == 0)
    F["i2v_cos_max"] = nan_if(imax, c_unknown)

    # ---------------- E. recency of interest ----------------
    for name, j in [("last_same_brand", BRAND), ("last_same_cate3", CATE3)]:
        lv = h_attr[last_idx, j][c_loc]
        cv = c_attr[:, j]
        F[name] = nan_if((cv == lv).astype(np.float64), (cv == -1) | (lv == -1) | ~has_h[c_loc])

    # ---------------- F. query-conditioned history ----------------
    q_terms = sess["query"].to_numpy(dtype=object)
    qt, ql = split_flat(pd.Series(q_terms), TERM_SEP)
    query_long = pd.DataFrame({"s": np.repeat(np.arange(nS), ql), "term": qt}).drop_duplicates()
    hq_long = pd.DataFrame({"r": np.repeat(np.flatnonzero(qm), q_lens), "s": q_rows, "term": q_toks})
    matched_rows = hq_long.merge(query_long, on=["s", "term"])["r"].unique()
    mt = np.zeros(nH, bool); mt[matched_rows] = True
    F["qhist_n_actions"] = np.bincount(h_loc[mt], minlength=nS)[c_loc].astype(np.float64)
    F["qhist_wid_cnt"] = np.nan_to_num(keyed_merge((c_loc, c_wid), (h_loc[mt], h_wid[mt]), np.ones(mt.sum()), "sum"))
    for name, j in [("qhist_brand_aff", BRAND), ("qhist_cate3_aff", CATE3)]:
        hv = h_attr[:, j]; mm = mt & (hv != -1)
        F[name] = nan_if(keyed_merge((c_loc, c_attr[:, j]), (h_loc[mm], hv[mm]), w[mm], "sum"), c_attr[:, j] == -1)

    # ---------------- G. training-set priors ----------------
    for k in FAMILIES["G"]:
        F[k] = pri[k].to_numpy(dtype=np.float64)

    # ---------------- H. position within the slate ----------------
    for f in H_BASE:
        x = pd.Series(F[f])
        grp = x.groupby(c_loc)
        F[f + "_slate_pct"] = grp.rank(ascending=False, pct=True).to_numpy()
        sd = grp.transform("std").replace(0, np.nan)
        F[f + "_slate_z"] = ((x - grp.transform("mean")) / sd).to_numpy()
    slate = np.bincount(c_loc, minlength=nS)
    F["slate_size"] = slate[c_loc].astype(np.float64)
    cv3 = c_attr[:, CATE3]
    same3 = pd.Series(1, index=pd.MultiIndex.from_arrays([c_loc, cv3])).groupby(level=[0, 1]).transform("sum").to_numpy()
    F["cate3_slate_share"] = nan_if(same3 / slate[c_loc], cv3 == -1)

    # ---------------- assemble ----------------
    out = pd.DataFrame({
        "session_id": cand["session_id"].to_numpy(),
        "wid": c_wid,
        "label": cand["label"].to_numpy(),
        "split": sess["split"].to_numpy()[c_loc],
    })
    feats = pd.DataFrame({k: np.asarray(F[k], dtype=np.float32) for k in FEATURE_COLUMNS})
    return pd.concat([out, feats], axis=1)

## Step 6 — Features from history cut at 7 and 90 days

For each window, this cell copies each chunk's history keeping only the actions newer than the cut-off, points `STAGE_DIR` at that copy (the candidates and priors are linked to the originals, since they do not depend on the history), runs `build_chunk_features` unchanged, and appends the result to the three split files of `processed/lgbm_features_window_{7,90}d/`, together with the group arrays and the feature list, in the same layout as the full-year files. The window features therefore differ from the full-year ones only in how much history they were computed from, which is what the time-window experiment in `Elysia_Results_and_Experiments.ipynb` (Section 7.1) needs. A window that is already built is skipped, so the cell is safe to rerun.

In [ ]:
import os, json, time
from collections import Counter
import numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq

ORIG_STAGE = STAGE_DIR

for MAX_DAYS in [7, 90]:
    out = OUTPUT_DIR / f"lgbm_features_window_{MAX_DAYS}d"
    out.mkdir(parents=True, exist_ok=True)
    if (out / "DONE.json").exists():
        print(f"{MAX_DAYS}-day window already built - skipping"); continue

    win_stage = LOCAL_DIR / f"stage_{MAX_DAYS}d"
    win_stage.mkdir(parents=True, exist_ok=True)
    writers, groups, counts = {}, {s: [] for s in ["train", "val", "test"]}, Counter()
    t0 = time.time()
    for i in range(N_CHUNKS):
        # history newer than the cut-off only; candidates and priors unchanged
        h = pd.read_parquet(ORIG_STAGE / f"hist_{i:03d}.parquet")
        h[h["days"] < MAX_DAYS].to_parquet(win_stage / f"hist_{i:03d}.parquet", index=False)
        for k in ["cand", "priors"]:
            dst = win_stage / f"{k}_{i:03d}.parquet"
            if not dst.exists():
                os.symlink((ORIG_STAGE / f"{k}_{i:03d}.parquet").resolve(), dst)

        STAGE_DIR = win_stage                    # build_chunk_features reads from STAGE_DIR
        try:
            feats = build_chunk_features(i)      # same function as the full-year features
        finally:
            STAGE_DIR = ORIG_STAGE
        feats[FEATURE_COLUMNS] = feats[FEATURE_COLUMNS].astype(np.float32)

        for s, part in feats.groupby("split", sort=False):
            part = part.drop(columns="split")
            sid = part["session_id"].to_numpy()
            cuts = np.flatnonzero(np.diff(sid)) + 1
            groups[s].append(np.diff(np.r_[0, cuts, len(sid)]))
            table = pa.Table.from_pandas(part, preserve_index=False)
            if s not in writers:
                writers[s] = pq.ParquetWriter(out / f"{s}.parquet", table.schema, compression="zstd")
            writers[s].write_table(table)
            counts[s] += len(part)
        (win_stage / f"hist_{i:03d}.parquet").unlink()
        del feats, h; gc.collect()
        if i % 10 == 0:
            print(f"{MAX_DAYS}d: chunk {i + 1}/{N_CHUNKS} ({(time.time() - t0) / 60:.1f} min)")

    for s, w in writers.items():
        w.close()
        np.save(out / f"{s}_groups.npy", np.concatenate(groups[s]).astype(np.int32))
    (out / "feature_list.json").write_text(json.dumps(FEATURE_COLUMNS))
    (out / "DONE.json").write_text(json.dumps(dict(counts)))
    print(f"{MAX_DAYS}-day window done:", dict(counts))

7d: chunk 1/86 (0.6 min)
7d: chunk 11/86 (2.7 min)
7d: chunk 21/86 (5.0 min)
7d: chunk 31/86 (7.1 min)
7d: chunk 41/86 (9.2 min)
7d: chunk 51/86 (11.4 min)
7d: chunk 61/86 (13.7 min)
7d: chunk 71/86 (15.9 min)
7d: chunk 81/86 (18.0 min)
7-day window done: {'val': 1536651, 'train': 12279873, 'test': 1531525}
90d: chunk 1/86 (0.5 min)
90d: chunk 11/86 (3.5 min)
90d: chunk 21/86 (6.9 min)
90d: chunk 31/86 (10.1 min)
90d: chunk 41/86 (13.3 min)
90d: chunk 51/86 (17.4 min)
90d: chunk 61/86 (20.5 min)
90d: chunk 71/86 (23.6 min)
90d: chunk 81/86 (27.0 min)
90-day window done: {'val': 1536651, 'train': 12279873, 'test': 1531525}
